In [1]:
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np

In [3]:
from peft import PeftModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_model_id = "mistralai/Mistral-7B-v0.1"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,  # Mistral, same as before
    quantization_config=bnb_config,  # Same quantization config as before
    device_map="auto",
    trust_remote_code=True,
)

eval_tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    add_bos_token=True,
    trust_remote_code=True,
)

ft_model = PeftModel.from_pretrained(base_model, "mistral-stockLM-finetune/checkpoint-1000")



ImportError: Using `bitsandbytes` 8-bit quantization requires Accelerate: `pip install accelerate` and the latest version of bitsandbytes: `pip install -i https://pypi.org/simple/ bitsandbytes`

In [4]:

def lm_gen(eval_prompt):
    model_input = eval_tokenizer(eval_prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        output = ft_model.generate(**model_input, pad_token_id=eval_tokenizer.eos_token_id, max_new_tokens=3, do_sample=True)
        content = eval_tokenizer.decode(output[0], skip_special_tokens=True)
        # print(eval_tokenizer.decode(output[0], skip_special_tokens=True))
    result_string  = content.replace(eval_prompt, "")
    return result_string 

In [5]:
def generate_prompt(t):
    full_prompt =f"""請給這篇文章打一個分數，分數介於-100～100之間，-100是負向，100是最好，0則是無法判斷

    ### Target sentence:
    {t}

    ### response:
    """
    return full_prompt

In [6]:

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

# stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]
stock_ids = ["2912"]

# 變異
irrs = []
bnhs = []
raw_data = []
for stock_id in stock_ids:
    print(f'執行{stock_id}')
    folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
    files = os.listdir(folder_path)
    sorted_files = sorted(files)

    # 載入買進價格
    df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
    price_np = df.to_numpy()
    prices = price_np.tolist()  # [時間,價格]

    signal_2 = []

    for price in prices:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(price[0], '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        sig = 0
        for index, file_name in enumerate(sorted_files):
            if file_name == price[0]:
                with open(folder_path + file_name, encoding='utf-8') as f:
                    news = f.read()
                    # lm的回應，回傳帶有買入訊號
                    sig = lm_gen(news)
                break

        signal_2.append([price[0], sig, price[1]])
        print(sig, "sd")


#     signal_2_np = np.array(signal_2)
#     # 回傳 IRR 以及總額投入
#     irr, bnh = calculate_irr(signal_2_np, 0)
#     irrs.append(irr)
#     bnhs.append(bnh)
#     raw_data.append(signal_2_np.tolist())
#     # print("每月內部報酬率", irr)
#     # print("總額投入報酬率", bnh)

# # 輸出結果
# write_folder = "evaluate_output/output"
# count = 1
# while True:
#     if os.path.exists(write_folder + str(count)):
#         count += 1
#         continue
#     else:
#         with open(write_folder + str(count), "a", encoding="UTF-8") as f:
#             content = (f'提示：{p_mutate_2}\n mutate from {p_mutate}\n'
#                         f'開始日期 {start_time.strftime("%Y%m%d")},'
#                         f'結束日期 {end_time.strftime("%Y%m%d")}\n')
#             for j in range(len(stock_ids)):
#                 content += (f'股票代號：{stock_ids[j]}\t'
#                             f'每月內部報酬率:{irrs[j]}\t'
#                             f'總額投入報酬率:{bnhs[j]}\n')
#             content += f'變異{i + 1}次，平均內部報酬率:{sum(irrs) / len(irrs)}，平均總額投入報酬率:{sum(bnhs) / len(bnhs)}\n)'
#             content += f'原始數據{raw_data}'
#             f.write(content)
#         print("寫入" + write_folder + str(count))
#         break


執行2912


NameError: name 'eval_tokenizer' is not defined